In [14]:
import pandas as pd
import sqlite3

# Loading Raw ACS Data into a DataFrame

In [15]:
df = pd.read_excel("../data/oo_raw.xlsx")

# Exploratory Data Analysis

Using a Custom Function from the Utilities Notebook

In [16]:
from utilities import basic_eda

basic_eda(df)

DataFrame Shape: (5245, 23)

 Column Names:
['Area Name', 'Area Type', 'SOC Title', 'Standard Occupational Classification (SOC)', 'SOC Major Group', 'SOC Classification', '2022 Estimated Employment', '2032 Projected Employment', 'Change', 'Percent Change', 'Annualized Percent Growth (Grow Rate)', 'Exits', 'Transfers', 'Openings', 'Mean Annual', 'Entry Annual', '25th Percentile Anual', 'Median Annual', '75th Percentile Annual', 'Experienced Annual', 'Typical Education Required for Entry', 'Typical Work Experience Required in Related Occupation', 'Typical On-the-Job Training Required to Achieve Competency']

 Data Types:
Area Name                                                      object
Area Type                                                      object
SOC Title                                                      object
Standard Occupational Classification (SOC)                      int64
SOC Major Group                                                 int64
SOC Classification     

# Cleaning

## Column Names

I will start by changing column names to descriptive ones in lower snake case.

In [17]:
# Create a column renaming dictionary
column_rename_map = {
    "Area Name": "area_name",
    "Area Type": "area_type",
    "SOC Title": "soc_title",
    "Standard Occupational Classification (SOC)": "occupation",
    "SOC Major Group": "soc_major_group",
    "SOC Classification": "soc_classification",
    "2022 Estimated Employment": "employment_2022",
    "2032 Projected Employment": "employment_2032",
    "Change": "employment_change",
    "Percent Change": "percent_change",
    "Annualized Percent Growth (Grow Rate)": "annualized_percent_growth",
    "Exits": "exits",
    "Transfers": "transfers",
    "Openings": "openings",
    "Mean Annual": "mean_annual_wage",
    "Entry Annual": "entry_annual_wage",
    "25th Percentile Anual": "percentile_25_wage",
    "Median Annual": "median_annual_wage",
    "75th Percentile Annual": "percentile_75_wage",
    "Experienced Annual": "experienced_annual_wage",
    "Typical Education Required for Entry": "education_required",
    "Typical Work Experience Required in Related Occupation": "work_experience_required",
    "Typical On-the-Job Training Required to Achieve Competency": "ojt_required"
}

# Apply the renaming
df = df.rename(columns=column_rename_map)

# Check the result
print(df.columns.tolist())


['area_name', 'area_type', 'soc_title', 'occupation', 'soc_major_group', 'soc_classification', 'employment_2022', 'employment_2032', 'employment_change', 'percent_change', 'annualized_percent_growth', 'exits', 'transfers', 'openings', 'mean_annual_wage', 'entry_annual_wage', 'percentile_25_wage', 'median_annual_wage', 'percentile_75_wage', 'experienced_annual_wage', 'education_required', 'work_experience_required', 'ojt_required']


## Handling Nulls

### Observations
1. The columns 'soc_classification', 'work_experience_required', and 'ojt_required' all have a large percentage of nulls.  
2. Several other columns have low percentages of nulls.

### Thoughts/Plan
1. Eliminate the 3 columns with nulls >10%. They are not necessary.
2. Fill nulls in the 'education_required' column to 'Not reported'.
3. Leave other nulls in Dataframe. 
    - These columns have null percentages <10/%.
    - Using a mean or median fill value for these columns would be misleading.
    - Nulls can be filtered during analysis.

In [18]:
columns_to_drop = [
    "soc_classification",
    "work_experience_required",
    "ojt_required"
]

df = df.drop(columns=columns_to_drop) # Eliminating columns with high percentage of nulls

df["education_required"] = df["education_required"].fillna("Not reported") # Replacing nulls in this column with "Not reported"


## Rechecking Data Types

In [19]:
df.dtypes

area_name                     object
area_type                     object
soc_title                     object
occupation                     int64
soc_major_group                int64
employment_2022                int64
employment_2032                int64
employment_change              int64
percent_change               float64
annualized_percent_growth    float64
exits                          int64
transfers                      int64
openings                       int64
mean_annual_wage             float64
entry_annual_wage            float64
percentile_25_wage           float64
median_annual_wage           float64
percentile_75_wage           float64
experienced_annual_wage      float64
education_required            object
dtype: object

# Converting to SQLite Database Table

In [20]:
# Connect to existing SQLite database
conn = sqlite3.connect("../data/cleaned_data.sqlite")

# Write the DataFrame `df` to the database as a new table called "puma_data"
df.to_sql(
    name="oo_data",       # name of the new table
    con=conn,               # database connection
    if_exists="replace",    # overwrite if table already exists
    index=False             # don't include the index as a column
)

# Close the connection
conn.close()

print("Table 'oo_data' successfully added to cleaned_data.sqlite.")

Table 'oo_data' successfully added to cleaned_data.sqlite.


In [21]:
df.head()

,area_name,area_type,soc_title,occupation,soc_major_group,employment_2022,employment_2032,employment_change,percent_change,annualized_percent_growth,exits,transfers,openings,mean_annual_wage,entry_annual_wage,percentile_25_wage,median_annual_wage,percentile_75_wage,experienced_annual_wage,education_required
0,Kentucky,State Level,Total All occupations,0,0,2049528,2146969,97441,4.7543,0.4656,1005096,1288925,2391462,54030.0,24880.0,31600.0,43730.0,62060.0,89810.0,Not reported
1,Kentucky,State Level,Management Occupations,110000,11,132055,142022,9967,7.5476,0.7303,38848,64583,113398,105980.0,43870.0,61360.0,91230.0,129120.0,176720.0,Not reported
2,Kentucky,State Level,Chief Executives,111011,11,3830,3536,-294,-7.6762,-0.7955,1138,1279,2123,220200.0,86710.0,120360.0,171990.0,NaN,NaN,Bachelor's degree
3,Kentucky,State Level,General and Operations Managers,111021,11,51299,54502,3203,6.2438,0.6075,13172,28413,44788,97500.0,36740.0,52440.0,78890.0,122510.0,172840.0,Bachelor's degree
4,Kentucky,State Level,Advertising and Promotions Managers,112011,11,457,468,11,2.4070,0.2381,96,308,415,86070.0,51490.0,67270.0,81600.0,101930.0,109600.0,Bachelor's degree
